In [3]:
from dotenv import load_dotenv
import os

load_dotenv("../.env")

os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("ARTICULOS_LANGSMITH")
os.environ["LANGCHAIN_PROJECT"] = "Autores de articulos"
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")
os.environ["GITHUB_TOKEN"] = os.getenv("GITHUB_TOKEN")
os.environ["GITHUB_PERSONAL_ACCESS_TOKEN"] = os.getenv("GITHUB_PERSONAL_ACCESS_TOKEN")
os.environ["NVIDIA_API_KEY"] = os.getenv("NVIDIA_API_KEY")
os.environ["ORCHESTRATOR_API_KEY_LOCAL"] = os.getenv("ORCHESTRATOR_API_KEY_LOCAL")
os.environ["ORCHESTRATOR_BASE_URL_LOCAL"] = os.getenv("ORCHESTRATOR_BASE_URL_LOCAL")

from typing import Annotated, TypedDict
from langchain_mcp_adapters.tools import load_mcp_tools
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langchain_groq import ChatGroq
from langgraph.graph.message import add_messages
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
import sqlite3
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver
from langchain_openai import ChatOpenAI

### State

In [4]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

### MCP Servers

In [5]:
system_env = dict(os.environ)

# MCP Client Configuration
client = MultiServerMCPClient(
    {
        "filesystem": {
            "command": "npx",
            "args": ["-y", "@modelcontextprotocol/server-filesystem", "/home/santi/Documentos/LangGraph/"],
            "transport": "stdio",
            "env": system_env
        },
    }
)

In [6]:
import httpx

# Crear cliente HTTP personalizado con el header correcto (X-API-Key)
def create_custom_http_client():
    """Cliente HTTP con autenticación via header X-API-Key"""
    return httpx.Client(
        headers={"X-API-Key": os.environ["ORCHESTRATOR_API_KEY_LOCAL"]},
        timeout=30.0,
    )

llm = ChatOpenAI(
    model="qwen2.5:7b", 
    temperature=0,
    base_url=os.environ["ORCHESTRATOR_BASE_URL_LOCAL"],
    api_key="dummy",  # No se usa, la autenticación va en el header
    http_client=create_custom_http_client(),  # Cliente personalizado
)

In [7]:
from langchain.agents import create_agent

async def run_agent():
    
    async with AsyncSqliteSaver.from_conn_string("filesystem_agent_history.db") as sqlite_saver:
        # 2. ENTER THE CONTEXT (The specific server room)
        # Instead of 'async with client', we use 'client.session("server_name")'
        async with client.session("filesystem") as session:
            
            # 3. LOAD TOOLS from that active session
            tools = await load_mcp_tools(session)
            # model = ChatGroq(model="llama-3.1-8b-instant", temperature=0).bind_tools(tools)      
            model = llm
            model = model.bind_tools(tools)
            # model = create_agent(llm, tools, verbose=True)

            print(f"Active tools: {[t.name for t in tools]}")

            base_path = "/home/santi/Documentos/LangGraph/"
            
            async def call_model(state: State):
                sys_msg = (
                    "system", 
                    "You are a technical assistant specialized in filesystem navigation.\n"
                    f"Your goal is to query and report the contents of the directory '{base_path}'.\n\n"
                    "Execution Instructions:\n"
                    "1. Review the message history of the current conversation.\n"
                    "2. If there is no previous message with tool results, call the tool to list files using exactly the indicated path.\n"
                    "3. If the files have already been listed in the history, do not—under any circumstances—invoke the tool again. Proceed directly to write a natural language summary of the files found and end the conversation.\n\n"
                    "Ensure you output ONLY the function call or the final text, never both at once."
                )
                
                prompt = [sys_msg] + state["messages"]
                response = model.invoke(prompt)

                return {"messages": [response]}

            # Graph Construction 
            workflow = StateGraph(State)
            workflow.add_node("agent", call_model)
            workflow.add_node("tools", ToolNode(tools)) # MCP tools are executed here

            workflow.add_edge(START, "agent")
            
            # Conditional logic to use tools
            def should_continue(state: State):
                if state["messages"][-1].tool_calls:
                    return "tools"
                return END

            workflow.add_conditional_edges("agent", should_continue)
            workflow.add_edge("tools", "agent")

            config = {"configurable": {"thread_id": "2"}}
            app = workflow.compile(checkpointer=sqlite_saver)

            # Execution
            inputs = {"messages": [("user", f"List the files in the directory {base_path}")]}        # IMPORTANT: The invocation occurs WITHIN the 'async with'
            result = await app.ainvoke(inputs, config=config) # Asynchronous graph invocation
            
            # for msg in result["messages"]:
            #     msg.pretty_print()

            # Listar el historial de estados
            async def print_history():
                print(f"--- Historial del Thread: {config['configurable']['thread_id']} ---")
                async for state in app.aget_state_history(config):
                    print(f"\nID: {state.config['configurable']['checkpoint_id']}")
                    print(f"Próximo nodo: {state.next}")
                    print(f"Mensajes: {len(state.values.get('messages', []))}")
                    print(f"Último mensaje: {state.values.get('messages')[-1].content if state.values.get('messages') else 'N/A'}")
                    print("-" * 40)

            await print_history()

# Execution in Notebook
await run_agent()

Active tools: ['read_file', 'read_text_file', 'read_media_file', 'read_multiple_files', 'write_file', 'edit_file', 'create_directory', 'list_directory', 'list_directory_with_sizes', 'directory_tree', 'move_file', 'search_files', 'get_file_info', 'list_allowed_directories']
--- Historial del Thread: 2 ---

ID: 1f152aa5-e1d4-6d75-802b-d582571348ba
Próximo nodo: ()
Mensajes: 26
Último mensaje: "The contents of the directory '/home/santi/Documentos/LangGraph/' include:\n\n- A `.env` file\n- A `.git` directory\n- A `.gitignore` file\n- A `.venv` directory\n- A `.vscode` directory\n- A `First Proyects` directory\n- A `MCP` directory\n- A `Multi-Services Router` directory\n- A `README_LangGraph.md` file\n- A `SOLUCION_API_AUTENTICACION.md` file\n- A `bots_report.txt` file\n- A `pyrightconfig.json` file\n- A `test.txt` file\n- A `test_api_fix.py` file"
----------------------------------------

ID: 1f152aa5-bd6f-62a0-802a-c1f772a0580b
Próximo nodo: ('agent',)
Mensajes: 25
Último mensaje: List t